## Pulling Gaming Subreddit Data and Conducting Sentiment Analysis
### Game: X

This Colab contains the pipeline used to 1. **Collect subreddit** data from the Cornell ConvoKit Library, and 2. Conduct **Sentiment Analysis** on the pulled data.

In [ ]:
try:
    import convokit
except ModuleNotFoundError:
    !pip install convokit
import nltk; nltk.download('punkt_tab')
!python3 -m spacy download en_core_web_sm

In [ ]:
import pandas as pd

In [ ]:
import torch
print(torch.cuda.is_available())

### Downloading corpus from ConvoKit 

In [ ]:
from convokit import Corpus, download

corpus_path = "subreddit-X"
corpus_name = "X"

In [ ]:
corpus = Corpus(download(corpus_path))

In [ ]:
print("Corpus Overview", )
print(("------------------------------"))

corpus.print_summary_stats()

print(("------------------------------\n"))

utt = corpus.random_utterance()

print("Random utterance")
print(("------------------------------"))

print("ID:", utt.id, "\n")
print("Reply_to:", utt.reply_to, "\n")
print("Timestamp:", utt.timestamp, "\n")
print("Text:", utt.text, "\n")
print("Conversation ID:", utt.conversation_id, "\n")
print("Speaker ID:", utt.speaker.id)
print(("------------------------------"))

### Minimal Preprocessing and Random Sampling

VADER and the fine tuned roBERTa model used in this study were created with social media data in mind. To ensure the models had access to the original structure of the text (VADER is case sensitive for example, using capitalization to collect information on magnituide, and fine tuned roBERTa uses embeddings which can be enriched by similar factors), preprocessing was minimal at this stage of the pipeline.

In [ ]:
# Remove posts (comments, posts) that have been deleted or removed from the subreddit corpus
# Also removes posts that are less than 10 characters

# NOTED IN THE LIMITATIONS, the filtering of removed and deleted reddit posts here is not very robust.
# This was found in hindsight after changes to the preprocessing pipeline were made, where originally before 
# this step some text cleaning was done, such as lower casing text, to ensure that simple removed and deleted
# posts would be removed.

def remove_invalid_posts(utt):
  if utt.text not in ["[deleted]", "[removed]", "removed"]:
    if len(utt.text) > 30:
      return utt

corpus = corpus.filter_utterances(corpus, remove_invalid_posts)

In [ ]:
import datetime

# Remove posts that might include interference from other games

# Starwars Battle Front - r/StarWarsBattlefront
# November 17, 2017 BF 2 was released

# Splatoon -- r/splatoon
# July 21, 2017

# Starwars Battle Front 2 Cuttoff - November 17, 2017
SWBFcutoff = datetime.datetime(2017, 11, 17).timestamp()

# Splatoon 2 Cutoff - July 21, 2017
SPTNcutoff = datetime.datetime(2017, 7, 21).timestamp()

def remove_interference_posts(utt):
    # cutoff = SWBFcutoff
    # cutoff = SPTNcutoff

    timestamp = utt.timestamp
    if utt.timestamp < cutoff:
        return utt

corpus = corpus.filter_utterances(corpus, remove_interference_posts)

In [ ]:
import random

sampled_convos = set(random.sample(list(corpus.conversations.keys()), 12500))

filtered_corpus = corpus.filter_utterances(corpus, lambda utt: utt.conversation_id in sampled_convos)

In [ ]:
print("Corpus Overview Post Processing", )
print(("------------------------------"))

corpus.print_summary_stats()

print(("------------------------------\n"))

utt = corpus.random_utterance()

### Creating Initial dataframes and VADER Sentiment Tagging

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

sia = SentimentIntensityAnalyzer()

In [ ]:
# Given VADER compund score, returns the final label in 
# acordance with VADER suggestions
def label_sentiment(compound):
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    else:
        return "neutral"

### Working at the Uterance Level
This section goes through the filtered corpus at the utterance level. Utterances are posts in the ConvoKit
library. This ensures individual posts, not entire threads are marked for sentiment.

In [ ]:
data_utterances = []

for utt in filtered_corpus.iter_utterances():
    text = utt.text

    if text:  
        scores = sia.polarity_scores(text)

        data_utterances.append({
            "utterance_id": utt.id,
            "speaker": utt.speaker.id if utt.speaker else None,
            "conversation_id": utt.conversation_id,
            "time": utt.timestamp,
            "text": text,

            # metadata fields, didn't end up getting used in the study
            "score": utt.meta.get("score") if utt.meta.get("score") else 0,
            "top_level_comment": utt.meta.get("top_level_comment") if utt.meta.get("top_level_comment") else None,

            "neg": scores["neg"],
            "neu": scores["neu"],
            "pos": scores["pos"],
            "compound": scores["compound"]
        })

df_utt_VADER = pd.DataFrame(data_utterances)

df_utt_VADER["VADER_sentiment_label"] = df_utt_VADER["compound"].apply(label_sentiment)

In [ ]:
df_utt_VADER.to_csv(f"{corpus_path}_utt.csv")

### Working at the Conversation Level
This section goes through the filtered corpus at the conversation level. Conversations are threads in the ConvoKit
library. The dataframe below goes through the filtered thread, and appends all the text from each post used in each thread to a single string. This will be helpful mostly for topic modeling. 

In [ ]:
data_convos = []

for convo in filtered_corpus.iter_conversations():

    utterances = convo.iter_utterances()

    texts = []

    for utt in utterances:
      texts.append(utt.text)

    final_text =  " ".join(texts)

    if final_text:
        scores = sia.polarity_scores(final_text)

        data_convos.append({
           
            "conversation_id": convo.id,
            "text": final_text,
            "neg": scores["neg"],
            "neu": scores["neu"],
            "pos": scores["pos"],
            "compound": scores["compound"]
        })

df_convo_VADER = pd.DataFrame(data_convos)

df_convo_VADER["VADER_sentiment_label"] = df_convo_VADER["compound"].apply(label_sentiment)

print(df_convo_VADER.head())

In [ ]:
df_convo_VADER.to_csv(f"{corpus_path}_convo.csv")

### Fine tuned roBERTa Sentiment Tagging

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm

# Sentiment pipeline using the RoBERTa-based cardiffnlp model
adv_sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512,
    device=0
)

In [ ]:
def analyze(batch):
    outputs = adv_sentiment_pipeline(batch["text"])
    return {
        "bert_label": [output["label"] for output in outputs],
        "bert_score": [output["score"] for output in outputs]
    }

### Utterance Level

In [ ]:
from datasets import Dataset

dataset_utt = Dataset.from_pandas(df_utt_VADER[["text"]])

dataset_utt = dataset_utt.map(
    analyze,
    batched=True,
    batch_size=128
)

In [ ]:
df_utt_VADER["bert_label"] = dataset_utt["bert_label"]
df_utt_VADER["bert_score"] = dataset_utt["bert_score"]

label_map = {
    "LABEL_0": -1,
    "LABEL_1": 0,
    "LABEL_2": 1
}

df_utt_VADER["bert_numeric"] = df_utt_VADER["bert_label"].map(label_map)

In [ ]:
df_utt_VADER.head()

In [ ]:
df_utt_VADER.to_csv(f"{corpus_path}_utt_final.csv", index=False)

### Conversation Level

In [ ]:
dataset_convo = Dataset.from_pandas(df_convo_VADER[["text"]])

dataset_convo = dataset_convo.map(
    analyze,
    batched=True,
    batch_size=64
)

In [ ]:
df_convo_VADER["bert_label"] = dataset_convo["bert_label"]
df_convo_VADER["bert_score"] = dataset_convo["bert_score"]

label_map = {
    "LABEL_0": -1,
    "LABEL_1": 0,
    "LABEL_2": 1
}

df_convo_VADER["bert_numeric"] = df_convo_VADER["bert_label"].map(label_map)

In [ ]:
df_convo_VADER.to_csv(f"{corpus_path}_convo_final.csv", index=False)